In [10]:
import os
import torch
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image

# ---------------------------------------------------------
# 1. MODEL MİMARİSİ (RGB Akışı ve Dropout)
# ---------------------------------------------------------
class RGBStream(nn.Module):
    def __init__(self, num_classes=2):
        super(RGBStream, self).__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        
        # İlk katmanları dondurma
        for param in self.backbone.parameters():
            param.requires_grad = False
            
        # Sadece son bloğu (layer4) eğitime açma
        for param in self.backbone.layer4.parameters():
            param.requires_grad = True
            
        # Sınıflandırıcıya DROPOUT ekleyerek aşırı ezberlemeyi önleme
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(p=0.5), # Nöronların %50'sini rastgele kapatır
            nn.Linear(num_features, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

# ---------------------------------------------------------
# 2. VERİ HAZIRLIĞI VE ÇEŞİTLENDİRME (Data Augmentation)
# ---------------------------------------------------------
class CASIADataset(Dataset):
    def __init__(self, root_dir):
        # Transform parametresini kaldırdık, çünkü bölme işleminden sonra uygulayacağız
        self.filepaths = []
        self.labels = []
        valid_extensions = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp')
        
        au_dir = os.path.join(root_dir, 'CASIA2', 'Au')
        if os.path.exists(au_dir):
            for img_name in os.listdir(au_dir):
                if img_name.lower().endswith(valid_extensions):
                    self.filepaths.append(os.path.join(au_dir, img_name))
                    self.labels.append(0) 
                
        tp_dir = os.path.join(root_dir, 'CASIA2', 'Tp')
        if os.path.exists(tp_dir):
            for img_name in os.listdir(tp_dir):
                if img_name.lower().endswith(valid_extensions):
                    self.filepaths.append(os.path.join(tp_dir, img_name))
                    self.labels.append(1) 

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img_path = self.filepaths[idx]
        image = Image.open(img_path).convert("RGB") 
        return image, self.labels[idx] # Sadece ham PIL görselini döndürür

# Dataset bölündükten sonra transform uygulamak için sarıcı sınıf
class ApplyTransform(Dataset):
    def __init__(self, dataset, transform):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        return self.transform(image), label

# EĞİTİM İÇİN DÖNÜŞÜMLER (Rastgele çevirme ve döndürme eklendi)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

# DOĞRULAMA İÇİN DÖNÜŞÜMLER (Sadece standart işlemler, resim değiştirilmez)
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

data_dir = '/kaggle/input/datasets/divg07/casia-20-image-tampering-detection-dataset'

# Veri setini ham olarak yükle
full_dataset = CASIADataset(root_dir=data_dir)
print(f"Toplam Yüklenen Görsel: {len(full_dataset)}")

# Veriyi Train (%80) ve Val (%20) olarak böl
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_subset, val_subset = random_split(full_dataset, [train_size, val_size])

# Transformları ilgili alt setlere (subset) uygula
train_dataset = ApplyTransform(train_subset, train_transforms)
val_dataset = ApplyTransform(val_subset, val_transforms)

print(f"Eğitim (Train) Seti: {len(train_dataset)}")
print(f"Doğrulama (Val) Seti: {len(val_dataset)}\n")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# ---------------------------------------------------------
# 3. EĞİTİM KURULUMU
# ---------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RGBStream(num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), 
    lr=1e-4 
)

# ---------------------------------------------------------
# 4. EĞİTİM VE DOĞRULAMA DÖNGÜSÜ (Best Model Kontrolü)
# ---------------------------------------------------------
num_epochs = 10
best_val_loss = float('inf') # En iyi modeli takip etmek için sonsuz ile başlatıyoruz
best_model_path = '/kaggle/working/best_rgb_model.pth'

print(f"Eğitim {device} cihazı üzerinde başlıyor...\n")

for epoch in range(num_epochs):
    # --- Eğitim Aşaması ---
    model.train() 
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    
    # --- Doğrulama (Validation) Aşaması ---
    model.eval() 
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad(): 
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    val_epoch_loss = val_loss / len(val_loader)
    val_epoch_acc = 100 * val_correct / val_total
    
    print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {epoch_loss:.4f}, Acc: %{epoch_acc:.2f} | Val Loss: {val_epoch_loss:.4f}, Acc: %{val_epoch_acc:.2f}")
    
    # --- En İyi Modeli Kaydetme ---
    if val_epoch_loss < best_val_loss:
        best_val_loss = val_epoch_loss
        torch.save(model.state_dict(), best_model_path)
        print(f"   --> Val Loss düştü! Yeni en iyi model kaydedildi.")

print(f"\nEğitim tamamlandı! En başarılı model ağırlıkları {best_model_path} konumunda.")

Toplam Yüklenen Görsel: 12614
Eğitim (Train) Seti: 10091
Doğrulama (Val) Seti: 2523

Eğitim cuda cihazı üzerinde başlıyor...

Epoch [1/10] | Train Loss: 0.5519, Acc: %70.56 | Val Loss: 0.4750, Acc: %76.22
   --> Val Loss düştü! Yeni en iyi model kaydedildi.
Epoch [2/10] | Train Loss: 0.4180, Acc: %80.04 | Val Loss: 0.4460, Acc: %78.80
   --> Val Loss düştü! Yeni en iyi model kaydedildi.
Epoch [3/10] | Train Loss: 0.3743, Acc: %81.79 | Val Loss: 0.4472, Acc: %77.80
Epoch [4/10] | Train Loss: 0.3433, Acc: %83.50 | Val Loss: 0.4608, Acc: %78.64


KeyboardInterrupt: 

In [11]:
import torch
import torch.optim as optim
import torch.nn as nn

print("--- AŞAMA 2: DERİN İNCE AYAR (DEEP FINE-TUNING) BAŞLIYOR ---\n")

# 1. En İyi Modeli Geri Yükleme
# Az önce %78.80 başarı ile kaydettiğimiz altın değerindeki modeli hafızaya alıyoruz.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RGBStream(num_classes=2).to(device)
model.load_state_dict(torch.load('/kaggle/working/best_rgb_model.pth'))
print("Önceki en iyi model başarıyla yüklendi.")

# 2. Katmanların Kilidini Açma (Unfreeze Layer 3 ve Layer 4)
# Önce güvenliğe alıp hepsini donduralım
for param in model.parameters():
    param.requires_grad = False

# Şimdi Layer 3, Layer 4 ve Sınıflandırıcıyı (fc) eğitime açalım
for param in model.backbone.layer3.parameters():
    param.requires_grad = True
for param in model.backbone.layer4.parameters():
    param.requires_grad = True
for param in model.backbone.fc.parameters():
    param.requires_grad = True

# 3. Yeni Optimizer ve Scheduler
# Öğrenme oranını çok küçültüyoruz (1e-5)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), 
    lr=1e-5 
)
criterion = nn.CrossEntropyLoss()

# Scheduler: Eğer 'val_loss' 2 epoch boyunca düşmezse (patience=2), öğrenme oranını yarıya indir (factor=0.5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# 4. İnce Ayar Eğitim Döngüsü
fine_tune_epochs = 15
best_ft_val_loss = float('inf')
best_ft_model_path = '/kaggle/working/best_rgb_finetuned.pth'

for epoch in range(fine_tune_epochs):
    # --- Eğitim Aşaması ---
    model.train() 
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    
    # --- Doğrulama (Validation) Aşaması ---
    model.eval() 
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad(): 
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    val_epoch_loss = val_loss / len(val_loader)
    val_epoch_acc = 100 * val_correct / val_total
    
    # Mevcut öğrenme oranını (LR) alalım ki ekranda görebilelim
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"Epoch [{epoch+1}/{fine_tune_epochs}] | LR: {current_lr:.1e} | Train Loss: {epoch_loss:.4f}, Acc: %{epoch_acc:.2f} | Val Loss: {val_epoch_loss:.4f}, Acc: %{val_epoch_acc:.2f}")
    
    # Scheduler'ı val_loss'a göre güncelle
    scheduler.step(val_epoch_loss)
    
    # En İyi Modeli Kaydetme
    if val_epoch_loss < best_ft_val_loss:
        best_ft_val_loss = val_epoch_loss
        torch.save(model.state_dict(), best_ft_model_path)
        print(f"   --> Harika! İnce Ayar Val Loss düştü. Yeni model '{best_ft_model_path}' olarak kaydedildi.")

print("\nDerin İnce Ayar (Deep Fine-Tuning) tamamlandı!")

--- AŞAMA 2: DERİN İNCE AYAR (DEEP FINE-TUNING) BAŞLIYOR ---

Önceki en iyi model başarıyla yüklendi.
Epoch [1/15] | LR: 1.0e-05 | Train Loss: 0.3488, Acc: %83.43 | Val Loss: 0.4462, Acc: %79.11
   --> Harika! İnce Ayar Val Loss düştü. Yeni model '/kaggle/working/best_rgb_finetuned.pth' olarak kaydedildi.
Epoch [2/15] | LR: 1.0e-05 | Train Loss: 0.3239, Acc: %84.08 | Val Loss: 0.4475, Acc: %79.11
Epoch [3/15] | LR: 1.0e-05 | Train Loss: 0.3154, Acc: %84.87 | Val Loss: 0.4578, Acc: %78.68
Epoch [4/15] | LR: 1.0e-05 | Train Loss: 0.2978, Acc: %85.76 | Val Loss: 0.4623, Acc: %79.39


KeyboardInterrupt: 

In [12]:
import os
import torch
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import numpy as np

print("--- 🕵️ GÜRÜLTÜ AKIŞI (SRM & NOISE STREAM) BAŞLATILIYOR ---\n")

# =====================================================================
# 1. VERİ HAZIRLIĞI (Sadece Au ve Tp Klasörlerini Okur)
# =====================================================================
class CASIADataset(Dataset):
    def __init__(self, root_dir):
        self.filepaths = []
        self.labels = []
        valid_extensions = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp')
        
        au_dir = os.path.join(root_dir, 'CASIA2', 'Au')
        if os.path.exists(au_dir):
            for img_name in os.listdir(au_dir):
                if img_name.lower().endswith(valid_extensions):
                    self.filepaths.append(os.path.join(au_dir, img_name))
                    self.labels.append(0) 
                
        tp_dir = os.path.join(root_dir, 'CASIA2', 'Tp')
        if os.path.exists(tp_dir):
            for img_name in os.listdir(tp_dir):
                if img_name.lower().endswith(valid_extensions):
                    self.filepaths.append(os.path.join(tp_dir, img_name))
                    self.labels.append(1) 

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img_path = self.filepaths[idx]
        image = Image.open(img_path).convert("RGB") 
        return image, self.labels[idx]

class ApplyTransform(Dataset):
    def __init__(self, dataset, transform):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        return self.transform(image), label

# =====================================================================
# 2. SRM FİLTRELERİ VE NOISE STREAM MİMARİSİ
# =====================================================================

# Sabit SRM Matrisleri (Görüntüdeki içeriği silip gürültüyü bırakır)
srm_filter1 = np.array([[0, 0, 0], [0, -1, 1], [0, 0, 0]], dtype=np.float32)
srm_filter2 = np.array([[0, 0, 0], [0, -1, 0], [0, 1, 0]], dtype=np.float32)
srm_filter3 = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32) / 4.0
srm_weights = np.zeros((3, 3, 3, 3), dtype=np.float32)

for i in range(3):
    srm_weights[0, i, :, :] = srm_filter1 
    srm_weights[1, i, :, :] = srm_filter2 
    srm_weights[2, i, :, :] = srm_filter3 

class SRMLayer(nn.Module):
    def __init__(self):
        super(SRMLayer, self).__init__()
        self.srm_conv = nn.Conv2d(3, 3, kernel_size=3, stride=1, padding=1, bias=False)
        self.srm_conv.weight = nn.Parameter(torch.from_numpy(srm_weights))
        
        # Filtreleri dondur (Ağın bunları değiştirmesini istemiyoruz)
        for param in self.srm_conv.parameters():
            param.requires_grad = False

    def forward(self, x):
        return self.srm_conv(x)

class NoiseStream(nn.Module):
    def __init__(self, num_classes=2):
        super(NoiseStream, self).__init__()
        
        # 1. Resim girer girmez SRM ile röntgenlenir
        self.srm_layer = SRMLayer()
        
        # 2. Röntgenlenen gürültü haritası ResNet'e verilir
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        
        # ResNet'in ilk katmanlarını dondur (Catastrophic Forgetting engellemesi)
        for param in self.backbone.parameters():
            param.requires_grad = False
            
        # Sadece son iki bloğu eğitime aç
        for param in self.backbone.layer3.parameters():
            param.requires_grad = True
        for param in self.backbone.layer4.parameters():
            param.requires_grad = True
            
        # Sınıflandırıcı (Dropout ile)
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(num_features, num_classes)
        )

    def forward(self, x):
        noise_map = self.srm_layer(x)
        return self.backbone(noise_map)

# =====================================================================
# 3. VERİ YÜKLEME VE DÖNÜŞÜMLER
# =====================================================================
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

data_dir = '/kaggle/input/datasets/divg07/casia-20-image-tampering-detection-dataset'
full_dataset = CASIADataset(root_dir=data_dir)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_subset, val_subset = random_split(full_dataset, [train_size, val_size])

train_dataset = ApplyTransform(train_subset, train_transforms)
val_dataset = ApplyTransform(val_subset, val_transforms)

# İşlemci çekirdeklerini kullanarak yüklemeyi hızlandıralım (num_workers=2)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

# =====================================================================
# 4. EĞİTİM DÖNGÜSÜ (NOISE STREAM)
# =====================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Eğitim Cihazı: {device} | Veri Seti Boyutu: {len(full_dataset)}\n")

model = NoiseStream(num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()

# Eğitim oranını (LR) derin öğrenmeye uygun başlatıyoruz
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

# Zeki LR Düşürücü (Val loss düşmezse LR'ı yarıya indirir)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

num_epochs = 15
best_val_loss = float('inf')
save_path = '/kaggle/working/best_noise_model.pth'

for epoch in range(num_epochs):
    model.train() 
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    
    model.eval() 
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad(): 
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    val_epoch_loss = val_loss / len(val_loader)
    val_epoch_acc = 100 * val_correct / val_total
    
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch+1}/{num_epochs}] | LR: {current_lr:.1e} | Train Loss: {epoch_loss:.4f}, Acc: %{epoch_acc:.2f} | Val Loss: {val_epoch_loss:.4f}, Acc: %{val_epoch_acc:.2f}")
    
    scheduler.step(val_epoch_loss)
    
    if val_epoch_loss < best_val_loss:
        best_val_loss = val_epoch_loss
        torch.save(model.state_dict(), save_path)
        print(f"   --> Val Loss düştü! Yeni Gürültü (Noise) modeli kaydedildi.")

print(f"\nEğitim tamamlandı. En başarılı SRM modeli ağırlıkları şurada: {save_path}")

--- 🕵️ GÜRÜLTÜ AKIŞI (SRM & NOISE STREAM) BAŞLATILIYOR ---

Eğitim Cihazı: cuda | Veri Seti Boyutu: 12614

Epoch [1/15] | LR: 1.0e-04 | Train Loss: 0.6237, Acc: %63.86 | Val Loss: 0.5806, Acc: %67.22
   --> Val Loss düştü! Yeni Gürültü (Noise) modeli kaydedildi.
Epoch [2/15] | LR: 1.0e-04 | Train Loss: 0.5622, Acc: %69.84 | Val Loss: 0.5582, Acc: %68.89
   --> Val Loss düştü! Yeni Gürültü (Noise) modeli kaydedildi.
Epoch [3/15] | LR: 1.0e-04 | Train Loss: 0.5203, Acc: %72.98 | Val Loss: 0.5536, Acc: %69.92
   --> Val Loss düştü! Yeni Gürültü (Noise) modeli kaydedildi.
Epoch [4/15] | LR: 1.0e-04 | Train Loss: 0.4866, Acc: %75.10 | Val Loss: 0.5815, Acc: %69.28
Epoch [5/15] | LR: 1.0e-04 | Train Loss: 0.4564, Acc: %77.40 | Val Loss: 0.5587, Acc: %70.23
Epoch [6/15] | LR: 1.0e-04 | Train Loss: 0.4330, Acc: %78.30 | Val Loss: 0.5856, Acc: %71.19
Epoch [7/15] | LR: 5.0e-05 | Train Loss: 0.3952, Acc: %80.71 | Val Loss: 0.5992, Acc: %70.71
Epoch [8/15] | LR: 5.0e-05 | Train Loss: 0.3796, Acc:

KeyboardInterrupt: 

In [13]:
import os
import torch
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import numpy as np

print("--- 🔗 BÜYÜK BİRLEŞME (TWO-STREAM FEATURE FUSION) BAŞLIYOR ---\n")

# =====================================================================
# 1. VERİ HAZIRLIĞI (Custom Dataset - Öncekiyle Aynı)
# =====================================================================
class CASIADataset(Dataset):
    def __init__(self, root_dir):
        self.filepaths = []
        self.labels = []
        valid_extensions = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp')
        
        au_dir = os.path.join(root_dir, 'CASIA2', 'Au')
        if os.path.exists(au_dir):
            for img_name in os.listdir(au_dir):
                if img_name.lower().endswith(valid_extensions):
                    self.filepaths.append(os.path.join(au_dir, img_name))
                    self.labels.append(0) 
                
        tp_dir = os.path.join(root_dir, 'CASIA2', 'Tp')
        if os.path.exists(tp_dir):
            for img_name in os.listdir(tp_dir):
                if img_name.lower().endswith(valid_extensions):
                    self.filepaths.append(os.path.join(tp_dir, img_name))
                    self.labels.append(1) 

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img_path = self.filepaths[idx]
        image = Image.open(img_path).convert("RGB") 
        return image, self.labels[idx]

class ApplyTransform(Dataset):
    def __init__(self, dataset, transform):
        self.dataset = dataset
        self.transform = transform
    def __len__(self): return len(self.dataset)
    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        return self.transform(image), label

# =====================================================================
# 2. ESKİ MİMARİLERİN TANIMLANMASI (Ağırlıkları yüklemek için)
# =====================================================================
class RGBStream(nn.Module):
    def __init__(self, num_classes=2):
        super(RGBStream, self).__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(nn.Dropout(p=0.5), nn.Linear(num_features, num_classes))
    def forward(self, x): return self.backbone(x)

# SRM Filtreleri
srm_filter1 = np.array([[0, 0, 0], [0, -1, 1], [0, 0, 0]], dtype=np.float32)
srm_filter2 = np.array([[0, 0, 0], [0, -1, 0], [0, 1, 0]], dtype=np.float32)
srm_filter3 = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32) / 4.0
srm_weights = np.zeros((3, 3, 3, 3), dtype=np.float32)
for i in range(3):
    srm_weights[0, i, :, :] = srm_filter1; srm_weights[1, i, :, :] = srm_filter2; srm_weights[2, i, :, :] = srm_filter3 

class SRMLayer(nn.Module):
    def __init__(self):
        super(SRMLayer, self).__init__()
        self.srm_conv = nn.Conv2d(3, 3, kernel_size=3, stride=1, padding=1, bias=False)
        self.srm_conv.weight = nn.Parameter(torch.from_numpy(srm_weights))
    def forward(self, x): return self.srm_conv(x)

class NoiseStream(nn.Module):
    def __init__(self, num_classes=2):
        super(NoiseStream, self).__init__()
        self.srm_layer = SRMLayer()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(nn.Dropout(p=0.5), nn.Linear(num_features, num_classes))
    def forward(self, x): return self.backbone(self.srm_layer(x))

# =====================================================================
# 3. YENİ BİRLEŞTİRME MİMARİSİ (TWO-STREAM FUSION)
# =====================================================================
class TwoStreamFusion(nn.Module):
    def __init__(self, rgb_model, noise_model, num_classes=2):
        super(TwoStreamFusion, self).__init__()
        
        # Önceden eğitilmiş modelleri içe aktarıyoruz
        self.rgb_stream = rgb_model
        self.noise_stream = noise_model
        
        # 1. KRİTİK ADIM: Modellerin son karar katmanlarını koparıp etkisiz eleman (Identity) ekliyoruz.
        # Artık bu ağlar sınıflandırma yapmayacak, sadece 2048'er adet özellik (feature) üretecek.
        self.rgb_stream.backbone.fc = nn.Identity()
        self.noise_stream.backbone.fc = nn.Identity()
        
        # 2. KRİTİK ADIM: Eski ağları "Eğitime Kapatıyoruz" (Freeze). 
        # Onlar zaten uzmanlaştı. Biz sadece ortak karar veren hakemi eğiteceğiz.
        for param in self.rgb_stream.parameters():
            param.requires_grad = False
        for param in self.noise_stream.parameters():
            param.requires_grad = False
            
        # 3. YENİ KARAR KATMANI: (2048 RGB) + (2048 Gürültü) = 4096 Özellik
        self.fusion_classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(2048 + 2048, 512), # Önce 512'ye sıkıştırıp analiz et
            nn.ReLU(),                   # Doğrusal olmayan ilişkileri bul
            nn.Dropout(p=0.3),
            nn.Linear(512, num_classes)  # Nihai Karar (Gerçek veya Sahte)
        )

    def forward(self, x):
        # Resmi aynı anda iki ajana da veriyoruz
        rgb_features = self.rgb_stream(x)     # Çıktı Boyutu: [Batch, 2048]
        noise_features = self.noise_stream(x) # Çıktı Boyutu: [Batch, 2048]
        
        # İki ajandan gelen özellikleri yanyana yapıştırıyoruz (Concatenation)
        fused_features = torch.cat((rgb_features, noise_features), dim=1) # Çıktı: [Batch, 4096]
        
        # Birleştirilmiş özellikleri hakeme (fusion_classifier) veriyoruz
        return self.fusion_classifier(fused_features)

# =====================================================================
# 4. ÇALIŞTIRMA VE EĞİTİM
# =====================================================================
if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print("1. Önceden Eğitilmiş Ajanlar Yükleniyor...")
    # RGB Modelini Yükle
    rgb_net = RGBStream(num_classes=2)
    rgb_net.load_state_dict(torch.load('/kaggle/working/best_rgb_finetuned.pth'))
    
    # Noise Modelini Yükle
    noise_net = NoiseStream(num_classes=2)
    noise_net.load_state_dict(torch.load('/kaggle/working/best_noise_model.pth'))
    
    print("2. Fusion Ağı Oluşturuluyor...")
    # İki uzmanı birleştiren ana modeli oluştur
    fusion_model = TwoStreamFusion(rgb_net, noise_net, num_classes=2).to(device)
    
    # Veri Yükleme (Data Augmentation)
    train_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
    ])
    val_transforms = transforms.Compose([
        transforms.Resize((224, 224)), 
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
    ])

    data_dir = '/kaggle/input/datasets/divg07/casia-20-image-tampering-detection-dataset'
    full_dataset = CASIADataset(root_dir=data_dir)

    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_subset, val_subset = random_split(full_dataset, [train_size, val_size])

    train_dataset = ApplyTransform(train_subset, train_transforms)
    val_dataset = ApplyTransform(val_subset, val_transforms)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

    # 3. SADECE YENİ KATMANIN EĞİTİMİ
    criterion = nn.CrossEntropyLoss()
    # Dikkat: Sadece requires_grad=True olanları (Yani sadece fusion_classifier'ı) eğitiyoruz
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, fusion_model.parameters()), lr=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    num_epochs = 15
    best_val_loss = float('inf')
    save_path = '/kaggle/working/best_fusion_model.pth'

    print(f"\n3. Eğitim Başlıyor! (Sadece Karar Katmanı Eğitilecek) - Cihaz: {device}\n")
    for epoch in range(num_epochs):
        fusion_model.train() 
        running_loss = 0.0; correct = 0; total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = fusion_model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100 * correct / total
        
        fusion_model.eval() 
        val_loss = 0.0; val_correct = 0; val_total = 0
        
        with torch.no_grad(): 
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = fusion_model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                
        val_epoch_loss = val_loss / len(val_loader)
        val_epoch_acc = 100 * val_correct / val_total
        
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch [{epoch+1}/{num_epochs}] | LR: {current_lr:.1e} | Train Loss: {epoch_loss:.4f}, Acc: %{epoch_acc:.2f} | Val Loss: {val_epoch_loss:.4f}, Acc: %{val_epoch_acc:.2f}")
        
        scheduler.step(val_epoch_loss)
        if val_epoch_loss < best_val_loss:
            best_val_loss = val_epoch_loss
            torch.save(fusion_model.state_dict(), save_path)
            print(f"   --> MUHTEŞEM! Fusion Val Loss düştü. Nihai model kaydedildi.")

--- 🔗 BÜYÜK BİRLEŞME (TWO-STREAM FEATURE FUSION) BAŞLIYOR ---

1. Önceden Eğitilmiş Ajanlar Yükleniyor...
2. Fusion Ağı Oluşturuluyor...

3. Eğitim Başlıyor! (Sadece Karar Katmanı Eğitilecek) - Cihaz: cuda

Epoch [1/15] | LR: 1.0e-03 | Train Loss: 0.3739, Acc: %82.27 | Val Loss: 0.3338, Acc: %83.91
   --> MUHTEŞEM! Fusion Val Loss düştü. Nihai model kaydedildi.
Epoch [2/15] | LR: 1.0e-03 | Train Loss: 0.3521, Acc: %83.05 | Val Loss: 0.3278, Acc: %84.23
   --> MUHTEŞEM! Fusion Val Loss düştü. Nihai model kaydedildi.
Epoch [3/15] | LR: 1.0e-03 | Train Loss: 0.3509, Acc: %83.45 | Val Loss: 0.3353, Acc: %83.31
Epoch [4/15] | LR: 1.0e-03 | Train Loss: 0.3526, Acc: %83.60 | Val Loss: 0.3321, Acc: %83.95
Epoch [5/15] | LR: 1.0e-03 | Train Loss: 0.3397, Acc: %83.60 | Val Loss: 0.3377, Acc: %82.96


KeyboardInterrupt: 

In [20]:
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output
import io

print("🕵️‍♂️ Canlı Test: Kendi resminizi seçerek modeli test edin!")

# =====================================================================
# 1. HAKEMİN KARAR FONKSİYONU (Eksik olan kısım eklendi)
# =====================================================================
def predict_image(image_path, model_path, device):
    print(f"Büyüteç altına alınan görsel: {image_path}")
    
    try:
        # Modeli Yükle (Mimari sınıflarının önceki hücrelerden hafızada olduğu varsayılır)
        rgb_net = RGBStream(num_classes=2)
        noise_net = NoiseStream(num_classes=2)
        model = TwoStreamFusion(rgb_net, noise_net, num_classes=2).to(device)
        
        # Eğittiğimiz şampiyon modeli yüklüyoruz
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.eval() 
        
        # Test Dönüşümleri
        test_transforms = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        image = Image.open(image_path).convert("RGB")
        image_tensor = test_transforms(image).unsqueeze(0).to(device)
        
        # Hakemin Kararı
        with torch.no_grad():
            outputs = model(image_tensor)
            probabilities = F.softmax(outputs, dim=1)
            confidence, predicted_class = torch.max(probabilities, 1)
            
        class_names = ["Orijinal (Au)", "Sahte / Manipüle Edilmiş (Tp)"]
        result_text = class_names[predicted_class.item()]
        conf_percentage = confidence.item() * 100
        
        print("-" * 50)
        print(f"Hakemin Kararı: {result_text}")
        print(f"Özgüven (Emin Olma Oranı): %{conf_percentage:.2f}")
        print("-" * 50)
        
    except Exception as e:
        print(f"Analiz sırasında bir hata oluştu: {e}")

# =====================================================================
# 2. ARAYÜZ VE YÜKLEME BUTONU
# =====================================================================
uploader = widgets.FileUpload(
    accept='.jpg, .jpeg, .png, .tif',
    multiple=False,
    description='Resim Seç',
    button_style='info'
)

out = widgets.Output()

def on_upload(change):
    with out:
        clear_output()
        try:
            # ipywidgets sürüm kontrolü
            if isinstance(uploader.value, tuple):
                content = uploader.value[0].content
            else:
                filename = list(uploader.value.keys())[0]
                content = uploader.value[filename]['content']
            
            # Resmi belleğe al
            image = Image.open(io.BytesIO(content)).convert("RGB")
            
            print("Yüklenen Görsel:")
            display(image.resize((250, 250))) 
            
            # Geçici olarak kaydet
            temp_path = '/kaggle/working/temp_uploaded_test.jpg'
            image.save(temp_path)
            
            print("\n🔄 Yapay Zeka analiz ediyor...\n")
            
            # Modeli çağır
            current_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            saved_fusion_model_path = "/kaggle/working/best_fusion_model.pth"
            
            predict_image(
                image_path=temp_path, 
                model_path=saved_fusion_model_path, 
                device=current_device
            )
            
            uploader.value.clear()
            
        except Exception as e:
            print(f"Dosya yükleme hatası: {e}")

uploader.observe(on_upload, names='value')
display(uploader, out)

🕵️‍♂️ Canlı Test: Kendi resminizi seçerek modeli test edin!


FileUpload(value=(), accept='.jpg, .jpeg, .png, .tif', button_style='info', description='Resim Seç')

Output()